In [1]:
# Анализ сохраненного диалога
print("🔍 Анализ сохраненного диалога\n")
print("=" * 70)

# Получить все сообщения из чата
all_messages = await working_memory.get_messages(chat_session_id)
print(f"\n💬 Всего сообщений в рабочей памяти: {len(all_messages)}")

if all_messages:
    print("\n📜 Последние 5 сообщений:")
    for msg in all_messages[-5:]:
        role_icon = "🧑" if msg['role'] == 'user' else "🤖"
        content_preview = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
        print(f"  {role_icon} [{msg['role']}]: {content_preview}")

# Получить созданные эпизоды
chat_episodes = await episodic_memory.query_episodes(
    agent_id=agent_id,
    filter_by_tags=["chat"],
    filter_by_success=True,
    sort_by="created_at",
    sort_order="desc",
    limit=10
)

print(f"\n📖 Эпизодов чата сохранено: {len(chat_episodes)}")
if chat_episodes:
    for ep in chat_episodes[:3]:
        print(f"  - {ep['outcome']} (важность: {ep['importance']})")

# Получить извлеченные знания
chat_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=agent_id,
    filter_by_tags=["chat"],
    sort_by="created_at",
    sort_order="desc",
    limit=10
)

print(f"\n🧠 Извлечено знаний: {len(chat_knowledge)}")
if chat_knowledge:
    for kb in chat_knowledge[:3]:
        print(f"  - {kb['knowledge'][:80]}... (уверенность: {kb['confidence']:.2f})")

# Итоговый контекст сессии
final_context = await working_memory.get_context(chat_session_id)
print(f"\n📊 Финальный контекст сессии:")
print(f"  - Всего сообщений: {final_context.get('total_messages', 0)}")
print(f"  - Последняя активность: {final_context.get('last_message_at', 'N/A')}")
print(f"  - Режим: {final_context.get('mode', 'N/A')}")

print("\n" + "=" * 70)
print("✅ Анализ завершен! Все данные диалога доступны через API памяти.")

🔍 Анализ сохраненного диалога



NameError: name 'working_memory' is not defined

In [2]:
# Интерактивный чат с агентом (с OpenAI API)
import re
import os
from datetime import datetime
from IPython.display import display, Markdown, clear_output

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI не установлен. Установите: pip install openai")

# Проверка API ключа
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key and OPENAI_AVAILABLE:
    print("⚠️  OPENAI_API_KEY не установлен в переменных окружения")
    openai_api_key = input("Введите ваш OpenAI API ключ (или оставьте пустым для fallback режима): ").strip()

# Инициализация OpenAI клиента
if OPENAI_AVAILABLE and openai_api_key:
    client = OpenAI(api_key=openai_api_key)
    USE_OPENAI = True
    print("✅ OpenAI API подключен!")
else:
    USE_OPENAI = False
    print("ℹ️  Используется fallback режим (без OpenAI)")

# Создаем новую сессию для чата
chat_session_id = f"chat_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Инициализируем сессию
await working_memory.create_session(
    session_id=chat_session_id,
    agent_id=agent_id,
    initial_context={
        "user_name": "Дмитрий",
        "chat_started_at": datetime.now().isoformat(),
        "mode": "interactive_chat",
        "topics_discussed": [],
        "use_openai": USE_OPENAI
    },
    ttl_seconds=7200  # 2 hours
)

print("\n🤖 Интерактивный чат-бот Memory Agents" + (" (powered by OpenAI)" if USE_OPENAI else " (Fallback mode)"))
print("=" * 70)
print("\n Привет! Я агент с памятью. Задавайте мне вопросы о Memory Agents,")
print("и я буду запоминать нашу беседу во все типы памяти!\n")
print("💡 Команды: 'exit' или 'quit' - завершить диалог")
print("           'memory' - показать статус памяти")
print("           'context' - показать текущий контекст")
print("           'history' - показать последние сообщения\n")
print("=" * 70)

# Счетчик сообщений
message_counter = 0
episode_id = f"ep_chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Системный промпт для агента
SYSTEM_PROMPT = """Ты - AI агент с продвинутой системой памяти Memory Agents. 

У тебя есть 5 типов памяти:
1. **Working Memory** (Redis) - текущий контекст и диалог
2. **Episodic Memory** (MongoDB) - эпизоды взаимодействий с траекториями
3. **Semantic Memory** (MongoDB + Qdrant) - долгосрочные знания с векторным поиском
4. **Procedural Memory** (MongoDB) - процедуры и паттерны с метриками
5. **Facts Memory** (MongoDB) - граф знаний о сущностях и связях

Ты помогаешь пользователю понять возможности Memory Agents.

Важные правила:
- Отвечай на русском языке
- Используй информацию из памяти (предоставлена в контексте)
- Будь дружелюбным и информативным
- Если знаешь что-то из памяти - упомяни это
- Давай конкретные примеры
- Не выдумывай информацию, которой нет в контексте
"""

async def generate_response_with_openai(user_message, context):
    """Генерация ответа через OpenAI API с контекстом из памяти"""
    
    # Получаем данные из памяти
    messages_history = await working_memory.get_messages(chat_session_id, limit=10)
    
    knowledge = await semantic_memory.query_knowledge(
        filter_by_agent_id=agent_id,
        min_confidence=0.7,
        limit=5
    )
    
    recent_episodes = await episodic_memory.query_episodes(
        agent_id=agent_id,
        filter_by_success=True,
        sort_by="created_at",
        sort_order="desc",
        limit=3
    )
    
    # Формируем контекст для LLM
    memory_context = f"""
📊 КОНТЕКСТ ИЗ ПАМЯТИ:

💬 История диалога ({len(messages_history)} сообщений):
{chr(10).join([f"- [{msg['role']}]: {msg['content'][:100]}" for msg in messages_history[-5:]])}

🧠 Знания из Semantic Memory ({len(knowledge)} единиц):
{chr(10).join([f"- {kb.get('knowledge', '')[:150]}... (confidence: {kb.get('confidence', 0):.2f})" for kb in knowledge[:3]])}

📖 Недавние эпизоды ({len(recent_episodes)} шт.):
{chr(10).join([f"- {ep.get('outcome', '')[:100]}" for ep in recent_episodes[:3]])}

📝 Обсужденные темы: {', '.join(context.get('topics_discussed', ['нет']))}
"""
    
    # Формируем сообщения для OpenAI
    openai_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "system", "content": memory_context}
    ]
    
    # Добавляем последние сообщения из истории
    for msg in messages_history[-6:]:
        openai_messages.append({
            "role": msg["role"],
            "content": msg["content"]
        })
    
    # Добавляем текущий вопрос пользователя
    openai_messages.append({
        "role": "user",
        "content": user_message
    })
    
    try:
        # Вызов OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # или gpt-3.5-turbo для экономии
            messages=openai_messages,
            temperature=0.7,
            max_tokens=500
        )
        
        answer = response.choices[0].message.content
        
        # Обновляем topics_discussed на основе ответа
        msg_lower = user_message.lower()
        topics = context.get('topics_discussed', [])
        
        if 'working' in msg_lower and 'working' not in topics:
            topics.append('working')
        if 'episodic' in msg_lower and 'episodic' not in topics:
            topics.append('episodic')
        if 'semantic' in msg_lower and 'semantic' not in topics:
            topics.append('semantic')
        if 'procedural' in msg_lower and 'procedural' not in topics:
            topics.append('procedural')
        if 'facts' in msg_lower and 'facts' not in topics:
            topics.append('facts')
        
        context['topics_discussed'] = topics
        
        return answer
        
    except Exception as e:
        print(f"\n⚠️  Ошибка OpenAI API: {e}")
        # Fallback на простой ответ
        return f"Извините, возникла ошибка при генерации ответа. Попробуйте снова или используйте fallback режим."

async def generate_response_fallback(user_message, context):
    """Fallback ответы без OpenAI"""
    msg_lower = user_message.lower()
    
    # Получаем базовую статистику
    messages = await working_memory.get_messages(chat_session_id, limit=10)
    knowledge = await semantic_memory.query_knowledge(
        filter_by_agent_id=agent_id,
        min_confidence=0.7,
        limit=3
    )
    
    topics = context.get('topics_discussed', [])
    
    # Простые ответы
    if any(word in msg_lower for word in ['привет', 'hello', 'hi']):
        return "Привет! Я Memory Agents чат-бот. Задавайте вопросы о системе памяти!"
    
    elif 'память' in msg_lower or 'memory' in msg_lower:
        stats = f"({len(messages)} сообщений, {len(knowledge)} знаний)"
        if 'working' in msg_lower:
            topics.append('working') if 'working' not in topics else None
            return f"Working Memory хранит текущий диалог в Redis. Сейчас {stats}"
        elif 'episodic' in msg_lower:
            topics.append('episodic') if 'episodic' not in topics else None
            return "Episodic Memory записывает эпизоды взаимодействий с траекториями в MongoDB"
        elif 'semantic' in msg_lower:
            topics.append('semantic') if 'semantic' not in topics else None
            return "Semantic Memory хранит знания с векторным поиском (MongoDB + Qdrant)"
        else:
            return f"Memory Agents имеет 5 типов памяти. Статистика: {stats}"
    
    else:
        return f"Интересный вопрос! У меня в памяти {len(knowledge)} знаний. Спросите о конкретном типе памяти?"

# Основной цикл чата
while True:
    try:
        # Получаем ввод пользователя
        user_input = input("\n🧑 Вы: ").strip()
        
        if not user_input:
            continue
        
        # Проверка команд
        if user_input.lower() in ['exit', 'quit', 'выход']:
            print("\n👋 Завершаю диалог...")
            break
        
        elif user_input.lower() == 'memory':
            msgs = await working_memory.get_messages(chat_session_id)
            episodes = await episodic_memory.query_episodes(agent_id=agent_id, limit=20)
            knowledge_items = await semantic_memory.query_knowledge(filter_by_agent_id=agent_id, limit=20)
            
            print(f"\n📊 Статус памяти:")
            print(f"  💬 Сообщений в диалоге: {len(msgs)}")
            print(f"  📖 Всего эпизодов: {len(episodes)}")
            print(f"  🧠 Всего знаний: {len(knowledge_items)}")
            
            ctx = await working_memory.get_context(chat_session_id)
            topics = ctx.get('topics_discussed', [])
            if topics:
                print(f"  📝 Обсужденные темы: {', '.join(topics)}")
            continue
        
        elif user_input.lower() == 'context':
            ctx = await working_memory.get_context(chat_session_id)
            print(f"\n📋 Текущий контекст:")
            for key, value in ctx.items():
                if key != 'topics_discussed':
                    print(f"  - {key}: {value}")
            topics = ctx.get('topics_discussed', [])
            if topics:
                print(f"  - topics_discussed: {', '.join(topics)}")
            continue
        
        elif user_input.lower() == 'history':
            msgs = await working_memory.get_messages(chat_session_id, limit=10)
            print(f"\n📜 История диалога ({len(msgs)} сообщений):")
            for msg in msgs[-5:]:
                role_icon = "🧑" if msg['role'] == 'user' else "🤖"
                print(f"  {role_icon} [{msg['role']}]: {msg['content'][:80]}...")
            continue
        
        message_counter += 1
        
        # 1. Сохранить сообщение пользователя
        await working_memory.append_message(
            session_id=chat_session_id,
            role="user",
            content=user_input,
            metadata={"message_number": message_counter}
        )
        
        # 2. Получить контекст и сгенерировать ответ
        context = await working_memory.get_context(chat_session_id)
        
        if USE_OPENAI:
            print("🤔 Думаю...", end='', flush=True)
            response = await generate_response_with_openai(user_input, context)
            print("\r", end='')  # Очистить "Думаю..."
        else:
            response = await generate_response_fallback(user_input, context)
        
        # 3. Сохранить ответ
        await working_memory.append_message(
            session_id=chat_session_id,
            role="assistant",
            content=response,
            metadata={"message_number": message_counter}
        )
        
        # 4. Обновить контекст
        await working_memory.update_context(
            session_id=chat_session_id,
            context_updates={
                "last_message_at": datetime.now().isoformat(),
                "total_messages": message_counter * 2,
                "last_topic": "chat interaction",
                "topics_discussed": context.get('topics_discussed', [])
            }
        )
        
        # 5. Создать эпизод каждые 3 сообщения
        if message_counter % 3 == 0:
            current_episode_id = f"{episode_id}_{message_counter // 3}"
            await episodic_memory.create_episode(
                episode_id=current_episode_id,
                episode_type=EpisodeType.INTERACTION,
                agent_id=agent_id,
                session_id=chat_session_id,
                context={
                    "messages_in_episode": 3,
                    "chat_mode": "interactive_openai" if USE_OPENAI else "interactive_fallback",
                    "topics_discussed": context.get('topics_discussed', [])
                },
                outcome=f"Обработано {message_counter} сообщений. Темы: {', '.join(context.get('topics_discussed', ['общие']))}",
                success=True,
                importance=0.6 + (len(context.get('topics_discussed', [])) * 0.05),
                user_satisfaction=0.85 if USE_OPENAI else 0.7,
                tags=["chat", "interactive", "openai" if USE_OPENAI else "fallback"] + context.get('topics_discussed', [])
            )
        
        # 6. Сохранить знание каждые 5 сообщений
        if message_counter % 5 == 0:
            topics_str = ', '.join(context.get('topics_discussed', ['различные темы']))
            knowledge_text = f"Интерактивный диалог с пользователем через {'OpenAI GPT-4' if USE_OPENAI else 'fallback систему'}. Обработано {message_counter} вопросов. Темы: {topics_str}."
            await semantic_memory.create_knowledge(
                knowledge_id=f"kb_chat_{message_counter}_{int(datetime.now().timestamp())}",
                knowledge=knowledge_text,
                source=SourceType.INFERRED,
                confidence=0.88 if USE_OPENAI else 0.75,
                agent_id=agent_id,
                tags=["chat", "openai" if USE_OPENAI else "fallback", "interaction"] + context.get('topics_discussed', []),
                temporal_scope="recent",
                half_life_days=30
            )
        
        # Вывести ответ
        print(f"\n🤖 Агент: {response}")
        
    except KeyboardInterrupt:
        print("\n\n⚠️  Диалог прерван (Ctrl+C)")
        break
    except Exception as e:
        print(f"\n❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()
        continue

# Финальная статистика
print("\n" + "=" * 70)
print("📊 Итоги диалога:")
print(f"  💬 Всего сообщений: {message_counter * 2}")
print(f"  📖 Создано эпизодов: {message_counter // 3}")
print(f"  🧠 Сохранено знаний: {message_counter // 5}")

final_context = await working_memory.get_context(chat_session_id)
topics = final_context.get('topics_discussed', [])
if topics:
    print(f"  📝 Обсуждено тем: {len(topics)} ({', '.join(topics)})")

print(f"  🔧 Режим: {'OpenAI GPT-4' if USE_OPENAI else 'Fallback'}")
print("\n✅ Вся беседа сохранена в памяти и доступна для последующего анализа!")
print("=" * 70)


⚠️  OpenAI не установлен. Установите: pip install openai
ℹ️  Используется fallback режим (без OpenAI)


NameError: name 'working_memory' is not defined

## 10. Интерактивный чат с агентом <a id='chat'></a>

Теперь вы можете пообщаться с агентом напрямую! Все сообщения будут автоматически записываться в память:

- 💬 **Working Memory** - хранит историю диалога
- 📖 **Episodic Memory** - создает эпизоды взаимодействия
- 🧠 **Semantic Memory** - извлекает и сохраняет знания
- 👤 **Facts Memory** - обновляет граф знаний о пользователе

Введите `exit` или `quit` для завершения диалога.

# Memory Agents - Полная демонстрация возможностей

Этот ноутбук демонстрирует все возможности библиотеки Memory Agents для создания AI агентов с различными типами памяти.

## Содержание

1. [Введение и настройка](#intro)
2. [Working Memory - Рабочая память](#working)
3. [Episodic Memory - Эпизодическая память](#episodic)
4. [Semantic Memory - Семантическая память](#semantic)
5. [Procedural Memory - Процедурная память](#procedural)
6. [Facts Memory - Память о фактах](#facts)
7. [Консолидация памяти](#consolidation)
8. [Интеграция всех типов памяти](#integration)
9. [Мониторинг и метрики](#monitoring)

## 1. Введение и настройка <a id='intro'></a>

Memory Agents предоставляет пять типов памяти для AI агентов:

- **Working Memory** - кратковременная рабочая память для текущего контекста
- **Episodic Memory** - память о событиях и эпизодах взаимодействия
- **Semantic Memory** - долговременная память знаний с временным затуханием
- **Procedural Memory** - память о процедурах, паттернах и навыках
- **Facts Memory** - память о фактах, сущностях и их связях

In [3]:
# Установка и импорты
import sys
import os
import asyncio
import json
import importlib
from pathlib import Path
from datetime import datetime, timedelta
from pprint import pprint


def detect_project_root(start_path: Path) -> Path:
    """Определить корень проекта по расположению пакета."""
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "memory_agents").exists() and (candidate / "domain").exists():
            return candidate
    return start_path


notebook_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
project_root = detect_project_root(notebook_dir)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

memory_agents_module = None
try:
    memory_agents_module = importlib.import_module("memory_agents")
except ImportError:
    pass

if memory_agents_module:
    WorkingMemoryService = memory_agents_module.WorkingMemoryService
    EpisodicMemoryService = memory_agents_module.EpisodicMemoryService
    SemanticMemoryService = memory_agents_module.SemanticMemoryService
    ProceduralMemoryService = memory_agents_module.ProceduralMemoryService
    FactsService = memory_agents_module.FactsService
    SourceType = memory_agents_module.SourceType
    EpisodeType = memory_agents_module.EpisodeType
else:
    # Fallback для запуска напрямую из репозитория без установки пакета
    from domain.memory.working import WorkingMemoryService
    from domain.memory.episodic import EpisodicMemoryService
    from domain.memory.semantic import SemanticMemoryService
    from domain.memory.procedural import ProceduralMemoryService
    from domain.memory.facts import FactsService
    from api.contracts.knowledge import SourceType
    from api.contracts.episode import EpisodeType

print("✅ Все модули успешно импортированы!")
print(f"🕐 Время запуска: {datetime.now()}")
print(f"📁 Корневая директория проекта: {project_root}")

ModuleNotFoundError: No module named 'domain'

In [ ]:
# Проверка подключения к сервисам (MongoDB, Redis, Qdrant, PostgreSQL)
# ВАЖНО: Убедитесь, что все сервисы запущены через docker-compose

async def check_services():
    """Проверка доступности всех сервисов"""
    print("🔍 Проверка подключения к сервисам...\n")
    
    # Проверка Working Memory (Redis)
    working_service = WorkingMemoryService()
    working_health = await working_service.health_check()
    print(f"Working Memory (Redis): {working_health['status']}")
    
    # Проверка Episodic Memory (MongoDB)
    episodic_service = EpisodicMemoryService()
    episodic_health = await episodic_service.health_check()
    print(f"Episodic Memory (MongoDB): {episodic_health['status']}")
    
    # Проверка Semantic Memory (MongoDB + Qdrant)
    semantic_service = SemanticMemoryService()
    semantic_health = await semantic_service.health_check()
    print(f"Semantic Memory (MongoDB + Qdrant): {semantic_health['status']}")
    
    # Проверка Procedural Memory (MongoDB)
    procedural_service = ProceduralMemoryService()
    procedural_health = await procedural_service.health_check()
    print(f"Procedural Memory (MongoDB): {procedural_health['status']}")
    
    # Проверка Facts Memory (PostgreSQL)
    facts_service = FactsService()
    facts_health = await facts_service.health_check()
    print(f"Facts Memory (PostgreSQL): {facts_health['status']}")
    
    print("\n✅ Все сервисы доступны!")

# Запуск проверки
await check_services()

Failed to create collection semantic_memory: BaseEventLoop.run_in_executor() got an unexpected keyword argument 'collection_name'
Failed to initialize Qdrant collections: BaseEventLoop.run_in_executor() got an unexpected keyword argument 'collection_name'
Failed to connect to Qdrant: BaseEventLoop.run_in_executor() got an unexpected keyword argument 'collection_name'
Episodic memory health check failed: BaseEventLoop.run_in_executor() got an unexpected keyword argument 'collection_name'


🔍 Проверка подключения к сервисам...

Working Memory (Redis): healthy
Episodic Memory (MongoDB): unhealthy
Semantic Memory (MongoDB + Qdrant): healthy
Procedural Memory (MongoDB): healthy
Facts Memory (PostgreSQL): healthy

✅ Все сервисы доступны!


## 2. Working Memory - Рабочая память <a id='working'></a>

Рабочая память используется для хранения краткосрочного контекста текущей сессии.
Идеально подходит для:
- Текущего диалога с пользователем
- Активных задач и целей
- Временных переменных состояния

In [10]:
# Инициализация Working Memory
working_memory = WorkingMemoryService()

# Создание сессии для агента
agent_id = "demo_agent_001"
session_id = "session_demo_2025"

print("📝 Создание сессии рабочей памяти...")

await working_memory.create_session(
    session_id=session_id,
    agent_id=agent_id,
    initial_context={
        "user_name": "Дмитрий",
        "task": "Демонстрация возможностей Memory Agents",
        "language": "Russian",
        "started_at": datetime.now().isoformat()
    },
    ttl_seconds=3600  # Сессия живет 1 час
)

print(f"✅ Сессия {session_id} создана для агента {agent_id}")

📝 Создание сессии рабочей памяти...
✅ Сессия session_demo_2025 создана для агента demo_agent_001


In [11]:
# Добавление сообщений в рабочую память
print("💬 Добавление сообщений в диалог...\n")

# Сообщение пользователя
await working_memory.append_message(
    session_id=session_id,
    role="user",
    content="Привет! Расскажи мне о возможностях Memory Agents.",
    metadata={"timestamp": datetime.now().isoformat()}
)

# Ответ ассистента
await working_memory.append_message(
    session_id=session_id,
    role="assistant",
    content="""Привет, Дмитрий! Memory Agents предоставляет 5 типов памяти:
    
1. Working Memory - для текущего контекста
2. Episodic Memory - для событий и эпизодов
3. Semantic Memory - для долговременных знаний
4. Procedural Memory - для процедур и паттернов
5. Facts Memory - для фактов и связей

Каждый тип оптимизирован для своих задач!""",
    metadata={"timestamp": datetime.now().isoformat()}
)

# Получение истории сообщений
messages = await working_memory.get_messages(
    session_id=session_id,
    limit=10
)

print("📜 История диалога:")
for msg in messages:
    print(f"\n[{msg['role'].upper()}]: {msg['content'][:100]}...")

💬 Добавление сообщений в диалог...

📜 История диалога:

[USER]: Привет! Расскажи мне о возможностях Memory Agents....

[ASSISTANT]: Привет, Дмитрий! Memory Agents предоставляет 5 типов памяти:

1. Working Memory - для текущего конте...


In [ ]:
# Управление контекстом сессии
print("🎯 Обновление контекста сессии...\n")

# Обновление контекста
await working_memory.update_context(
    session_id=session_id,
    context_updates={
        "current_topic": "Working Memory",
        "messages_count": 2,
        "user_engagement": "high"
    }
)

# Получение текущего контекста
context = await working_memory.get_context(session_id=session_id)
print("📊 Текущий контекст сессии:")
pprint(context)

🎯 Обновление контекста сессии...

📊 Текущий контекст сессии:
{'agent_id': 'demo_agent_001',
 'current_topic': 'Working Memory',
 'language': 'Russian',
 'messages_count': 2,
 'started_at': '2025-10-28T18:37:45.755951',
 'task': 'Демонстрация возможностей Memory Agents',
 'user_engagement': 'high',
 'user_name': 'Дмитрий'}


In [ ]:
# Метрики рабочей памяти
metrics = await working_memory.get_metrics()
print("📈 Метрики Working Memory:")
pprint(metrics)

Failed to get working memory metrics: 'RedisClient' object has no attribute 'execute'


📈 Метрики Working Memory:
{'error': "'RedisClient' object has no attribute 'execute'",
 'timestamp': '2025-10-30T19:44:06.703246'}


## 3. Episodic Memory - Эпизодическая память <a id='episodic'></a>

Эпизодическая память хранит события и эпизоды взаимодействия.
Идеально подходит для:
- Истории взаимодействий с пользователем
- Траекторий выполнения задач
- Анализа успехов и неудач

In [ ]:
# Инициализация Episodic Memory
episodic_memory = EpisodicMemoryService()

print("📖 Создание эпизодов в памяти...\n")

# Создание успешного эпизода
episode_1_id = await episodic_memory.create_episode(
    episode_id="ep_demo_001",
    episode_type=EpisodeType.INTERACTION,
    agent_id=agent_id,
    session_id=session_id,
    context={
        "user_name": "Дмитрий",
        "topic": "Знакомство с Memory Agents"
    },
    outcome="Пользователь успешно познакомился с концепцией типов памяти",
    success=True,
    importance=0.8,
    user_satisfaction=0.9,
    tags=["introduction", "successful", "educational"]
)

print(f"✅ Создан эпизод: {episode_1_id}")

# Добавление траектории к эпизоду
await episodic_memory.append_to_trajectory(
    episode_id="ep_demo_001",
    step_data={
        "step": 1,
        "action": "greeting",
        "description": "Приветствие пользователя",
        "timestamp": datetime.now().isoformat()
    }
)

await episodic_memory.append_to_trajectory(
    episode_id="ep_demo_001",
    step_data={
        "step": 2,
        "action": "explain_memory_types",
        "description": "Объяснение 5 типов памяти",
        "timestamp": datetime.now().isoformat()
    }
)

await episodic_memory.append_to_trajectory(
    episode_id="ep_demo_001",
    step_data={
        "step": 3,
        "action": "receive_positive_feedback",
        "description": "Получение положительной обратной связи",
        "timestamp": datetime.now().isoformat()
    }
)

print("✅ Траектория эпизода обновлена (3 шага)")

📖 Создание эпизодов в памяти...

✅ Создан эпизод: 6903c00e3dd98f3dd09bd12c
✅ Траектория эпизода обновлена (3 шага)


In [ ]:
# Создание еще нескольких эпизодов для демонстрации
print("📖 Создание дополнительных эпизодов...\n")

# Эпизод с частичным успехом
await episodic_memory.create_episode(
    episode_id="ep_demo_002",
    episode_type=EpisodeType.TASK,
    agent_id=agent_id,
    session_id=session_id,
    context={
        "task": "Настройка подключения к базам данных",
        "difficulty": "medium"
    },
    outcome="Подключение настроено, но потребовалось несколько попыток",
    success=True,
    importance=0.6,
    user_satisfaction=0.7,
    tags=["setup", "database", "technical"]
)

# Неуспешный эпизод (для обучения)
await episodic_memory.create_episode(
    episode_id="ep_demo_003",
    episode_type=EpisodeType.ERROR,
    agent_id=agent_id,
    session_id=session_id,
    context={
        "error_type": "ConnectionError",
        "service": "Qdrant"
    },
    outcome="Не удалось подключиться к Qdrant из-за неправильного порта",
    success=False,
    importance=0.7,
    user_satisfaction=0.3,
    tags=["error", "qdrant", "connection"]
)

print("✅ Создано 3 эпизода разных типов")

📖 Создание дополнительных эпизодов...

✅ Создано 3 эпизода разных типов


In [ ]:
# Поиск эпизодов по различным критериям
print("🔍 Поиск эпизодов в памяти...\n")

# Поиск всех успешных эпизодов
successful_episodes = await episodic_memory.query_episodes(
    agent_id=agent_id,
    filter_by_success=True,
    sort_by="importance",
    sort_order="desc"
)

print(f"✅ Найдено {len(successful_episodes)} успешных эпизодов:")
for ep in successful_episodes:
    print(f"  - {ep['episode_id']}: {ep['outcome'][:60]}... (важность: {ep['importance']})")

# Поиск эпизодов с ошибками
error_episodes = await episodic_memory.query_episodes(
    agent_id=agent_id,
    filter_by_episode_type=[EpisodeType.ERROR],
    limit=10
)

print(f"\n❌ Найдено {len(error_episodes)} эпизодов с ошибками:")
for ep in error_episodes:
    print(f"  - {ep['episode_id']}: {ep['outcome'][:60]}...")

🔍 Поиск эпизодов в памяти...

✅ Найдено 50 успешных эпизодов:
  - ep_integrated_demo: Успешно обработан запрос с использованием всех типов памяти... (важность: 0.95)
  - ep_integrated_demo: Успешно обработан запрос с использованием всех типов памяти... (важность: 0.95)
  - ep_integrated_demo: Успешно обработан запрос с использованием всех типов памяти... (важность: 0.95)
  - ep_integrated_demo: Успешно обработан запрос с использованием всех типов памяти... (важность: 0.95)
  - ep_integrated_demo: Успешно обработан запрос с использованием всех типов памяти... (важность: 0.95)
  - ep_consolidation_4: Успешно объяснил концепцию консолидации памяти (попытка 5)... (важность: 0.8999999999999999)
  - ep_consolidation_4: Успешно объяснил концепцию консолидации памяти (попытка 5)... (важность: 0.8999999999999999)
  - ep_consolidation_4: Успешно объяснил концепцию консолидации памяти (попытка 5)... (важность: 0.8999999999999999)
  - ep_consolidation_4: Успешно объяснил концепцию консолидации пам

In [ ]:
# Получение полной информации об эпизоде с траекторией
episode_full = await episodic_memory.get_episode(
    episode_id="ep_demo_001",
    include_trajectory=True
)

print("📊 Детали эпизода ep_demo_001:")
print(f"\nТип: {episode_full['episode_type']}")
print(f"Результат: {episode_full['outcome']}")
print(f"Успех: {'✅' if episode_full['success'] else '❌'}")
print(f"Важность: {episode_full['importance']}")
print(f"Удовлетворенность: {episode_full['user_satisfaction']}")
print(f"\nТраектория ({len(episode_full['trajectory'])} шагов):")
for step in episode_full['trajectory']:
    print(f"  {step['step']}. {step['action']}: {step['description']}")

📊 Детали эпизода ep_demo_001:

Тип: interaction
Результат: Пользователь успешно познакомился с концепцией типов памяти
Успех: ✅
Важность: 0.8
Удовлетворенность: 0.9

Траектория (36 шагов):
  1. greeting: Приветствие пользователя
  2. explain_memory_types: Объяснение 5 типов памяти
  3. receive_positive_feedback: Получение положительной обратной связи
  1. greeting: Приветствие пользователя
  2. explain_memory_types: Объяснение 5 типов памяти
  3. receive_positive_feedback: Получение положительной обратной связи
  1. greeting: Приветствие пользователя
  2. explain_memory_types: Объяснение 5 типов памяти
  3. receive_positive_feedback: Получение положительной обратной связи
  1. greeting: Приветствие пользователя
  2. explain_memory_types: Объяснение 5 типов памяти
  3. receive_positive_feedback: Получение положительной обратной связи
  1. greeting: Приветствие пользователя
  2. explain_memory_types: Объяснение 5 типов памяти
  3. receive_positive_feedback: Получение положительной обратн

## 4. Semantic Memory - Семантическая память <a id='semantic'></a>

Семантическая память хранит долгосрочные знания с поддержкой векторного поиска.
Идеально подходит для:
- Накопления знаний из опыта
- Семантического поиска по контексту
- Временного затухания устаревших знаний

In [ ]:
# Инициализация Semantic Memory
semantic_memory = SemanticMemoryService()

print("🧠 Создание знаний в семантической памяти...\n")

# Создание знания из взаимодействия с пользователем
knowledge_1 = await semantic_memory.create_knowledge(
    knowledge_id="kb_demo_001",
    knowledge="Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русском языке",
    source=SourceType.USER_PROVIDED,
    confidence=0.95,
    agent_id=agent_id,
    tags=["user_preference", "language", "communication_style"],
    temporal_scope="always",
    half_life_days=365  # Знание актуально год
)

print(f"✅ Создано знание: {knowledge_1}")

# Создание технического знания
knowledge_2 = await semantic_memory.create_knowledge(
    knowledge_id="kb_demo_002",
    knowledge="""Memory Agents использует следующие хранилища:
    - Redis для Working Memory
    - MongoDB для Episodic, Semantic и Procedural Memory
    - Qdrant для векторного поиска в Semantic Memory
    - PostgreSQL для Facts Memory с поддержкой графов""",
    source=SourceType.LEARNED,
    confidence=1.0,
    agent_id=agent_id,
    tags=["architecture", "databases", "technical"],
    temporal_scope="always",
    half_life_days=180
)

print(f"✅ Создано знание: {knowledge_2}")

# Создание временного знания
knowledge_3 = await semantic_memory.create_knowledge(
    knowledge_id="kb_demo_003",
    knowledge="Текущая демонстрация проходит в формате Jupyter Notebook",
    source=SourceType.INFERRED,
    confidence=0.9,
    agent_id=agent_id,
    tags=["context", "session", "format"],
    temporal_scope="recent",
    half_life_days=7  # Актуально только неделю
)

print(f"✅ Создано знание: {knowledge_3}")

🧠 Создание знаний в семантической памяти...

✅ Создано знание: 6903c0423dd98f3dd09bd12f
✅ Создано знание: 6903c0423dd98f3dd09bd130
✅ Создано знание: 6903c0423dd98f3dd09bd131


In [ ]:
# Поиск знаний по различным критериям
print("🔍 Поиск знаний в семантической памяти...\n")

# Поиск по источнику
user_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=agent_id,
    filter_by_source=[SourceType.USER_PROVIDED],
    sort_by="confidence",
    sort_order="desc"
)

print(f"📚 Знания от пользователя ({len(user_knowledge)}):")
for kb in user_knowledge:
    print(f"  - {kb['knowledge'][:80]}... (уверенность: {kb['confidence']})")

# Поиск по тегам
technical_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=agent_id,
    filter_by_tags=["technical"],
    min_confidence=0.8
)

print(f"\n⚙️ Техническая информация ({len(technical_knowledge)}):")
for kb in technical_knowledge:
    print(f"  - {kb['knowledge'][:80]}...")

🔍 Поиск знаний в семантической памяти...

📚 Знания от пользователя (9):
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.98)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с примерами кода на русск... (уверенность: 0.95)
  - Пользователь Дмитрий предпочитает детальные объяснения с при

In [ ]:
# Обновление знания (например, изменение уверенности)
print("♻️ Обновление существующего знания...\n")

updated = await semantic_memory.update_knowledge(
    knowledge_id="kb_demo_001",
    updates={
        "confidence": 0.98,  # Повысили уверенность
        "tags": ["user_preference", "language", "communication_style", "verified"]
    }
)

if updated:
    print("✅ Знание успешно обновлено")
    updated_kb = await semantic_memory.get_knowledge("kb_demo_001")
    print(f"Новая уверенность: {updated_kb['confidence']}")
    print(f"Теги: {updated_kb['tags']}")

♻️ Обновление существующего знания...

✅ Знание успешно обновлено
Новая уверенность: 0.98
Теги: ['user_preference', 'language', 'communication_style', 'verified']


## 5. Procedural Memory - Процедурная память <a id='procedural'></a>

Процедурная память хранит паттерны, процедуры и навыки.
Идеально подходит для:
- Хранения последовательностей действий
- Оптимизации часто используемых процедур
- Адаптации стратегий на основе успеха

In [ ]:
# Инициализация Procedural Memory
procedural_memory = ProceduralMemoryService()

print("⚙️ Создание процедур в памяти...\n")

# Создание процедуры для обработки вопросов пользователя
procedure_1 = await procedural_memory.create_procedure(
    procedure_id="proc_demo_001",
    name="Обработка вопроса пользователя",
    description="Стандартная процедура ответа на вопрос пользователя с примерами",
    agent_id=agent_id,
    steps=[
        {
            "step_number": 1,
            "action": "parse_question",
            "description": "Разобрать вопрос и определить тип запроса",
            "parameters": {"extract_intent": True, "identify_entities": True}
        },
        {
            "step_number": 2,
            "action": "search_knowledge",
            "description": "Найти релевантную информацию в семантической памяти",
            "parameters": {"use_vector_search": True, "min_confidence": 0.7}
        },
        {
            "step_number": 3,
            "action": "generate_response",
            "description": "Сформировать ответ с примерами кода",
            "parameters": {"include_examples": True, "language": "Russian"}
        },
        {
            "step_number": 4,
            "action": "validate_response",
            "description": "Проверить качество ответа",
            "parameters": {"check_completeness": True}
        }
    ],
    tags=["user_interaction", "question_answering"],
    initial_success_rate=0.85
)

print(f"✅ Создана процедура: {procedure_1}")

# Создание процедуры обработки ошибок
procedure_2 = await procedural_memory.create_procedure(
    procedure_id="proc_demo_002",
    name="Обработка ошибок подключения",
    description="Процедура диагностики и исправления ошибок подключения к сервисам",
    agent_id=agent_id,
    steps=[
        {
            "step_number": 1,
            "action": "identify_error",
            "description": "Определить тип ошибки и затронутый сервис",
            "parameters": {"parse_stack_trace": True}
        },
        {
            "step_number": 2,
            "action": "check_service_health",
            "description": "Проверить статус сервиса",
            "parameters": {"timeout": 5}
        },
        {
            "step_number": 3,
            "action": "retry_connection",
            "description": "Попытаться переподключиться",
            "parameters": {"max_retries": 3, "backoff_multiplier": 2}
        },
        {
            "step_number": 4,
            "action": "log_error",
            "description": "Записать ошибку в эпизодическую память",
            "parameters": {"include_context": True}
        }
    ],
    tags=["error_handling", "connection", "diagnostics"],
    initial_success_rate=0.75
)

print(f"✅ Создана процедура: {procedure_2}")

Procedure proc_demo_001 already exists, returning existing ID
Procedure proc_demo_002 already exists, returning existing ID


⚙️ Создание процедур в памяти...

✅ Создана процедура: 6900ca8fd61c2172eaafa63c
✅ Создана процедура: 6900ca8fd61c2172eaafa63d


In [ ]:
# Запись выполнения процедуры
print("▶️ Запись выполнения процедуры...\n")

execution_1 = await procedural_memory.record_execution(
    procedure_id="proc_demo_001",
    success=True,
    execution_time_ms=250.5,
    context={
        "question": "Как работает Working Memory?",
        "response_quality": "high"
    },
    outcome="Успешно ответил на вопрос с примерами кода"
)

print(f"✅ Выполнение записано: {execution_1}")

# Еще несколько выполнений для накопления статистики
await procedural_memory.record_execution(
    procedure_id="proc_demo_001",
    success=True,
    execution_time_ms=180.3,
    context={"question": "Что такое Episodic Memory?"},
    outcome="Успешный ответ"
)

await procedural_memory.record_execution(
    procedure_id="proc_demo_001",
    success=False,
    execution_time_ms=420.1,
    context={"question": "Непонятный вопрос"},
    outcome="Не удалось определить intent"
)

print("✅ Записано 3 выполнения процедуры")

▶️ Запись выполнения процедуры...

✅ Выполнение записано: proc_demo_001_exec_1761655065.900487
✅ Записано 3 выполнения процедуры


In [ ]:
# Получение статистики процедуры
procedure_full = await procedural_memory.get_procedure(
    procedure_id="proc_demo_001",
    include_executions=True
)

print("📊 Статистика процедуры 'Обработка вопроса пользователя':")
print(f"\nНазвание: {procedure_full['name']}")
print(f"Шагов: {len(procedure_full['steps'])}")
print(f"Выполнений: {procedure_full['execution_count']}")
print(f"Успешность: {procedure_full['success_rate']:.1%}")
print(f"Среднее время: {procedure_full['avg_execution_time_ms']:.1f} мс")
print(f"\nПоследние выполнения:")
for exec in procedure_full['executions'][:3]:
    status = "✅" if exec['success'] else "❌"
    print(f"  {status} {exec['execution_time_ms']:.1f}мс - {exec['outcome']}")

📊 Статистика процедуры 'Обработка вопроса пользователя':

Название: Обработка вопроса пользователя
Шагов: 4
Выполнений: 18
Успешность: 66.7%
Среднее время: 283.6 мс

Последние выполнения:
  ✅ 250.5мс - Успешно ответил на вопрос с примерами кода
  ✅ 180.3мс - Успешный ответ
  ❌ 420.1мс - Не удалось определить intent


In [ ]:
# Поиск наиболее успешных процедур
print("🏆 Поиск лучших процедур...\n")

top_procedures = await procedural_memory.query_procedures(
    agent_id=agent_id,
    sort_by="success_rate",
    sort_order="desc",
    limit=5
)

print("Топ процедур по успешности:")
for i, proc in enumerate(top_procedures, 1):
    print(f"{i}. {proc['name']} - {proc['success_rate']:.1%} ({proc['execution_count']} выполнений)")

🏆 Поиск лучших процедур...

Топ процедур по успешности:
1. Обработка ошибок подключения - 75.0% (0 выполнений)
2. Обработка вопроса пользователя - 66.7% (18 выполнений)


## 6. Facts Memory - Память о фактах <a id='facts'></a>

Facts Memory хранит структурированные факты о сущностях и их связях.
Идеально подходит для:
- Графа знаний о сущностях
- Атрибутов и характеристик
- Связей между объектами

In [ ]:
# Инициализация Facts Memory
facts_memory = FactsService()

print("🗂️ Создание сущностей и фактов...\n")

# Создание сущности "Пользователь"
user_entity = await facts_memory.create_entity(
    entity_id="user_dmitry",
    entity_type="person",
    name="Дмитрий",
    agent_id=agent_id,
    attributes={
        "role": "developer",
        "language": "Russian",
        "expertise": ["AI", "Python", "Memory Systems"],
        "preference_detail_level": "high"
    }
)

print(f"✅ Создана сущность: {user_entity}")

# Создание сущности "Memory Agents"
system_entity = await facts_memory.create_entity(
    entity_id="system_memory_agents",
    entity_type="software_system",
    name="Memory Agents",
    agent_id=agent_id,
    attributes={
        "version": "1.0.0",
        "memory_types": 5,
        "databases": ["MongoDB", "Redis", "PostgreSQL", "Qdrant"],
        "architecture": "layered"
    }
)

print(f"✅ Создана сущность: {system_entity}")

# Создание сущностей для каждого типа памяти
memory_types = [
    ("memory_working", "Working Memory", {"storage": "Redis", "ttl": "session-based"}),
    ("memory_episodic", "Episodic Memory", {"storage": "MongoDB", "type": "events"}),
    ("memory_semantic", "Semantic Memory", {"storage": "MongoDB+Qdrant", "type": "knowledge"}),
    ("memory_procedural", "Procedural Memory", {"storage": "MongoDB", "type": "procedures"}),
    ("memory_facts", "Facts Memory", {"storage": "PostgreSQL", "type": "facts"})
]

for entity_id, name, attrs in memory_types:
    await facts_memory.create_entity(
        entity_id=entity_id,
        entity_type="memory_component",
        name=name,
        agent_id=agent_id,
        attributes=attrs
    )

print(f"✅ Создано {len(memory_types)} компонентов памяти")

🗂️ Создание сущностей и фактов...

✅ Создана сущность: user_dmitry
✅ Создана сущность: system_memory_agents
✅ Создано 5 компонентов памяти


In [ ]:
# Создание связей между сущностями
print("🔗 Создание связей между сущностями...\n")

# Связь: Пользователь использует систему
relation_1 = await facts_memory.create_relation(
    source_entity_id="user_dmitry",
    relation_type="uses",
    target_entity_id="system_memory_agents",
    agent_id=agent_id,
    properties={
        "since": "2025-10-18",
        "purpose": "learning and demonstration",
        "frequency": "daily"
    },
    confidence=1.0
)

print(f"✅ Создана связь: Дмитрий -> uses -> Memory Agents")

# Связи: Система содержит компоненты памяти
for entity_id, name, _ in memory_types:
    await facts_memory.create_relation(
        source_entity_id="system_memory_agents",
        relation_type="contains",
        target_entity_id=entity_id,
        agent_id=agent_id,
        properties={"required": True},
        confidence=1.0
    )

print(f"✅ Создано 5 связей: Memory Agents -> contains -> [типы памяти]")

# Создание связи зависимости между компонентами
await facts_memory.create_relation(
    source_entity_id="memory_episodic",
    relation_type="feeds_into",
    target_entity_id="memory_semantic",
    agent_id=agent_id,
    properties={
        "process": "consolidation",
        "description": "Episodes are consolidated into knowledge"
    },
    confidence=1.0
)

print(f"✅ Создана связь: Episodic -> feeds_into -> Semantic")

🔗 Создание связей между сущностями...

✅ Создана связь: Дмитрий -> uses -> Memory Agents
✅ Создано 5 связей: Memory Agents -> contains -> [типы памяти]
✅ Создана связь: Episodic -> feeds_into -> Semantic


In [ ]:
# Создание фактов о сущностях
print("📌 Создание фактов...\n")

# Факт о пользователе
fact_1 = await facts_memory.create_fact(
    entity_id="user_dmitry",
    attribute="current_learning_topic",
    value="Memory Agents demonstration",
    agent_id=agent_id,
    confidence=1.0,
    source="session_context"
)

print(f"✅ Факт: Дмитрий изучает Memory Agents")

# Технические факты о системе
await facts_memory.create_fact(
    entity_id="system_memory_agents",
    attribute="python_version_required",
    value="3.8+",
    agent_id=agent_id,
    confidence=1.0,
    source="documentation"
)

await facts_memory.create_fact(
    entity_id="system_memory_agents",
    attribute="deployment_method",
    value="docker-compose",
    agent_id=agent_id,
    confidence=1.0,
    source="configuration"
)

print(f"✅ Факты о системе добавлены")

📌 Создание фактов...

✅ Факт: Дмитрий изучает Memory Agents
✅ Факты о системе добавлены


In [ ]:
# Запросы к графу знаний
print("🔍 Исследование графа знаний...\n")

# Получить полную информацию о пользователе
user_info = await facts_memory.get_entity(
    entity_id="user_dmitry",
    include_relations=True,
    include_facts=True
)

print("👤 Информация о пользователе:")
print(f"Имя: {user_info['name']}")
print(f"Тип: {user_info['entity_type']}")
print(f"\nАтрибуты:")
for key, value in user_info['attributes'].items():
    print(f"  - {key}: {value}")

print(f"\nСвязи ({len(user_info['relations'])}):")
for rel in user_info['relations']:
    print(f"  - {rel['relation_type']} -> {rel['target_entity_id']}")

print(f"\nФакты ({len(user_info['facts'])}):")
for fact in user_info['facts']:
    print(f"  - {fact['attribute']}: {fact['value']}")

🔍 Исследование графа знаний...

👤 Информация о пользователе:
Имя: Дмитрий
Тип: person

Атрибуты:
  - role: developer
  - language: Russian
  - expertise: ['AI', 'Python', 'Memory Systems']
  - preference_detail_level: high

Связи (5):
  - uses -> system_memory_agents
  - uses -> system_memory_agents
  - uses -> system_memory_agents
  - uses -> system_memory_agents
  - uses -> system_memory_agents

Факты (5):
  - current_learning_topic: Memory Agents demonstration
  - current_learning_topic: Memory Agents demonstration
  - current_learning_topic: Memory Agents demonstration
  - current_learning_topic: Memory Agents demonstration
  - current_learning_topic: Memory Agents demonstration


In [ ]:
# Поиск связанных сущностей
print("🔗 Поиск связей в графе...\n")

# Найти все компоненты системы
system_components = await facts_memory.query_relations(
    agent_id=agent_id,
    source_entity_id="system_memory_agents",
    relation_type="contains"
)

print(f"Компоненты Memory Agents ({len(system_components)}):")
for comp in system_components:
    # Получить детали компонента
    component = await facts_memory.get_entity(comp['target_entity_id'])
    storage = component['attributes'].get('storage', 'unknown')
    print(f"  - {component['name']} (хранилище: {storage})")

🔗 Поиск связей в графе...

Компоненты Memory Agents (25):
  - Working Memory (хранилище: Redis)
  - Episodic Memory (хранилище: MongoDB)
  - Semantic Memory (хранилище: MongoDB+Qdrant)
  - Procedural Memory (хранилище: MongoDB)
  - Facts Memory (хранилище: PostgreSQL)
  - Working Memory (хранилище: Redis)
  - Episodic Memory (хранилище: MongoDB)
  - Semantic Memory (хранилище: MongoDB+Qdrant)
  - Procedural Memory (хранилище: MongoDB)
  - Facts Memory (хранилище: PostgreSQL)
  - Working Memory (хранилище: Redis)
  - Episodic Memory (хранилище: MongoDB)
  - Semantic Memory (хранилище: MongoDB+Qdrant)
  - Procedural Memory (хранилище: MongoDB)
  - Facts Memory (хранилище: PostgreSQL)
  - Working Memory (хранилище: Redis)
  - Episodic Memory (хранилище: MongoDB)
  - Semantic Memory (хранилище: MongoDB+Qdrant)
  - Procedural Memory (хранилище: MongoDB)
  - Facts Memory (хранилище: PostgreSQL)
  - Working Memory (хранилище: Redis)
  - Episodic Memory (хранилище: MongoDB)
  - Semantic Memory

## 7. Консолидация памяти <a id='consolidation'></a>

Один из ключевых процессов - консолидация эпизодов в семантическую память.
Система автоматически:
- Кластеризует похожие эпизоды
- Извлекает общие паттерны
- Создает знания высокого уровня

In [ ]:
# Создание дополнительных эпизодов для демонстрации консолидации
print("📖 Создание серии похожих эпизодов...\n")

# Серия успешных взаимодействий на одну тему
for i in range(5):
    await episodic_memory.create_episode(
        episode_id=f"ep_consolidation_{i}",
        episode_type=EpisodeType.INTERACTION,
        agent_id=agent_id,
        session_id=session_id,
        context={
            "topic": "memory_consolidation",
            "user_question_type": "technical"
        },
        outcome=f"Успешно объяснил концепцию консолидации памяти (попытка {i+1})",
        success=True,
        importance=0.7 + (i * 0.05),
        user_satisfaction=0.8 + (i * 0.03),
        tags=["consolidation", "explanation", "successful"]
    )

print("✅ Создано 5 похожих эпизодов")

📖 Создание серии похожих эпизодов...

✅ Создано 5 похожих эпизодов


In [ ]:
# Запуск консолидации старых эпизодов
print("🔄 Запуск консолидации эпизодов в знания...\n")

# Получить старые эпизоды для консолидации
old_episodes = await episodic_memory.query_episodes(
    agent_id=agent_id,
    filter_by_consolidated=False,  # Только не консолидированные
    time_range_end=datetime.now(),
    limit=20
)

print(f"Найдено {len(old_episodes)} эпизодов для консолидации")

if len(old_episodes) > 0:
    # Запустить консолидацию
    consolidation_result = await episodic_memory.consolidate_old_episodes(
        agent_id=agent_id,
        max_age_days=0,  # Консолидировать все эпизоды
        min_importance=0.0,
        limit=20
    )
    
    print("\n📊 Результаты консолидации:")
    print(f"  Обработано эпизодов: {consolidation_result['episodes_processed']}")
    print(f"  Найдено кластеров: {consolidation_result['clusters_found']}")
    print(f"  Создано знаний: {consolidation_result['knowledge_items_created']}")
    
    if consolidation_result['knowledge_items_created'] > 0:
        knowledge_ids = consolidation_result['consolidation_details']['created_knowledge_ids']
        print(f"\n💡 Созданные знания:")
        for kb_id in knowledge_ids[:3]:  # Показать первые 3
            kb = await semantic_memory.get_knowledge(kb_id)
            if kb:
                print(f"  - {kb['knowledge'][:100]}...")
                print(f"    Уверенность: {kb['confidence']:.2f}")
else:
    print("\n⚠️ Нет эпизодов для консолидации")

🔄 Запуск консолидации эпизодов в знания...

Найдено 20 эпизодов для консолидации

📊 Результаты консолидации:
  Обработано эпизодов: 71
  Найдено кластеров: 6
  Создано знаний: 6

💡 Созданные знания:


## 8. Интеграция всех типов памяти <a id='integration'></a>

Демонстрация того, как все типы памяти работают вместе.

In [ ]:
# Сценарий: обработка сложного запроса с использованием всех типов памяти
print("🎯 Комплексный сценарий: обработка запроса пользователя\n")
print("="*70)

# 1. Working Memory: получить текущий контекст
print("\n1️⃣ Working Memory: Получение текущего контекста сессии")
current_context = await working_memory.get_context(session_id)
print(f"   Пользователь: {current_context.get('user_name', 'Unknown')}")
print(f"   Текущая тема: {current_context.get('current_topic', 'None')}")
print(f"   Сообщений в сессии: {current_context.get('messages_count', 0)}")

# 2. Facts Memory: получить информацию о пользователе
print("\n2️⃣ Facts Memory: Получение профиля пользователя")
user_profile = await facts_memory.get_entity("user_dmitry")
preferences = user_profile['attributes']
print(f"   Язык: {preferences.get('language', 'Unknown')}")
print(f"   Уровень детализации: {preferences.get('preference_detail_level', 'medium')}")
print(f"   Экспертиза: {', '.join(preferences.get('expertise', []))}")

# 3. Semantic Memory: поиск релевантных знаний
print("\n3️⃣ Semantic Memory: Поиск релевантных знаний")
relevant_knowledge = await semantic_memory.query_knowledge(
    filter_by_agent_id=agent_id,
    filter_by_tags=["technical"],
    min_confidence=0.7,
    sort_by="confidence",
    limit=3
)
print(f"   Найдено знаний: {len(relevant_knowledge)}")
for kb in relevant_knowledge[:2]:
    print(f"   - {kb['knowledge'][:80]}... (conf: {kb['confidence']:.2f})")

# 4. Procedural Memory: выбор подходящей процедуры
print("\n4️⃣ Procedural Memory: Выбор процедуры обработки")
best_procedure = await procedural_memory.query_procedures(
    agent_id=agent_id,
    filter_by_tags=["user_interaction"],
    sort_by="success_rate",
    limit=1
)
if best_procedure:
    proc = best_procedure[0]
    print(f"   Выбрана: {proc['name']}")
    print(f"   Успешность: {proc['success_rate']:.1%}")
    print(f"   Шагов: {len(proc['steps'])}")

# 5. Episodic Memory: анализ похожих прошлых ситуаций
print("\n5️⃣ Episodic Memory: Поиск похожих прошлых эпизодов")
similar_episodes = await episodic_memory.query_episodes(
    agent_id=agent_id,
    filter_by_tags=["successful"],
    filter_by_success=True,
    sort_by="user_satisfaction",
    limit=3
)
print(f"   Найдено похожих эпизодов: {len(similar_episodes)}")
for ep in similar_episodes[:2]:
    print(f"   - {ep['outcome'][:70]}... (satisfaction: {ep['user_satisfaction']:.2f})")

# 6. Создание нового эпизода на основе обработки
print("\n6️⃣ Episodic Memory: Создание эпизода обработки запроса")
new_episode = await episodic_memory.create_episode(
    episode_id="ep_integrated_demo",
    episode_type=EpisodeType.TASK,
    agent_id=agent_id,
    session_id=session_id,
    context={
        "integrated_systems": ["working", "facts", "semantic", "procedural", "episodic"],
        "user": user_profile['name'],
        "knowledge_used": len(relevant_knowledge),
        "procedure": proc['name'] if best_procedure else None
    },
    outcome="Успешно обработан запрос с использованием всех типов памяти",
    success=True,
    importance=0.95,
    user_satisfaction=0.92,
    tags=["integration", "demonstration", "all_memory_types"]
)
print(f"   ✅ Эпизод создан: {new_episode}")

print("\n" + "="*70)
print("✅ Комплексный сценарий завершен успешно!")

🎯 Комплексный сценарий: обработка запроса пользователя


1️⃣ Working Memory: Получение текущего контекста сессии
   Пользователь: Дмитрий
   Текущая тема: Working Memory
   Сообщений в сессии: 2

2️⃣ Facts Memory: Получение профиля пользователя
   Язык: Russian
   Уровень детализации: high
   Экспертиза: AI, Python, Memory Systems

3️⃣ Semantic Memory: Поиск релевантных знаний
   Найдено знаний: 3
   - Memory Agents использует следующие хранилища:
    - Redis для Working Memory
   ... (conf: 1.00)
   - Memory Agents использует следующие хранилища:
    - Redis для Working Memory
   ... (conf: 1.00)

4️⃣ Procedural Memory: Выбор процедуры обработки
   Выбрана: Обработка вопроса пользователя
   Успешность: 66.7%
   Шагов: 4

5️⃣ Episodic Memory: Поиск похожих прошлых эпизодов
   Найдено похожих эпизодов: 3
   - Успешно объяснил концепцию консолидации памяти (попытка 5)... (satisfaction: 0.92)
   - Успешно объяснил концепцию консолидации памяти (попытка 5)... (satisfaction: 0.92)

6️⃣ Epis

## 9. Мониторинг и метрики <a id='monitoring'></a>

Система предоставляет детальные метрики для каждого типа памяти.

In [ ]:
# Сбор метрик со всех типов памяти
print("📊 Сбор метрик системы...\n")
print("="*70)

# Working Memory метрики
print("\n🔹 Working Memory")
working_metrics = await working_memory.get_metrics()
print(f"   Активных сессий: {working_metrics.get('active_sessions', 0)}")
print(f"   Всего сообщений: {working_metrics.get('total_messages', 0)}")

# Episodic Memory метрики
print("\n🔹 Episodic Memory")
episodic_metrics = await episodic_memory.get_metrics()
print(f"   Всего эпизодов: {episodic_metrics.get('total_episodes', 0)}")
print(f"   Успешных эпизодов: {episodic_metrics.get('successful_episodes', 0)}")
print(f"   Средняя важность: {episodic_metrics.get('avg_importance', 0):.2f}")
print(f"   Средняя удовлетворенность: {episodic_metrics.get('avg_user_satisfaction', 0):.2f}")

episodes_by_type = episodic_metrics.get('episodes_by_type', {})
if episodes_by_type:
    print("   По типам:")
    for ep_type, count in episodes_by_type.items():
        print(f"     - {ep_type}: {count}")

# Semantic Memory метрики
print("\n🔹 Semantic Memory")
semantic_metrics = await semantic_memory.get_metrics()
print(f"   Всего знаний: {semantic_metrics.get('total_knowledge', 0)}")
print(f"   Высокая уверенность (>0.8): {semantic_metrics.get('high_confidence_knowledge', 0)}")

kb_by_source = semantic_metrics.get('knowledge_by_source', {})
if kb_by_source:
    print("   По источникам:")
    for source, count in kb_by_source.items():
        print(f"     - {source}: {count}")

# Procedural Memory метрики
print("\n🔹 Procedural Memory")
procedural_metrics = await procedural_memory.get_metrics()
print(f"   Всего процедур: {procedural_metrics.get('total_procedures', 0)}")
print(f"   Всего выполнений: {procedural_metrics.get('total_executions', 0)}")
print(f"   Средняя успешность: {procedural_metrics.get('avg_success_rate', 0):.1%}")
print(f"   Среднее время выполнения: {procedural_metrics.get('avg_execution_time', 0):.1f} мс")

# Facts Memory метрики
print("\n🔹 Facts Memory")
facts_metrics = await facts_memory.get_metrics()
print(f"   Всего сущностей: {facts_metrics.get('total_entities', 0)}")
print(f"   Всего связей: {facts_metrics.get('total_relations', 0)}")
print(f"   Всего фактов: {facts_metrics.get('total_facts', 0)}")

entities_by_type = facts_metrics.get('entities_by_type', {})
if entities_by_type:
    print("   Сущности по типам:")
    for ent_type, count in entities_by_type.items():
        print(f"     - {ent_type}: {count}")

print("\n" + "="*70)

Failed to get working memory metrics: 'RedisClient' object has no attribute 'execute'


📊 Сбор метрик системы...


🔹 Working Memory
   Активных сессий: 0
   Всего сообщений: 0

🔹 Episodic Memory
   Всего эпизодов: 71
   Успешных эпизодов: 0
   Средняя важность: 0.00
   Средняя удовлетворенность: 0.00

🔹 Semantic Memory
   Всего знаний: 52
   Высокая уверенность (>0.8): 48
   По источникам:
     - user_stated: 9
     - inferred: 19
     - observed: 0
     - consolidated: 24
     - system: 0

🔹 Procedural Memory
   Всего процедур: 2
   Всего выполнений: 0
   Средняя успешность: 0.0%
   Среднее время выполнения: 0.0 мс

🔹 Facts Memory
   Всего сущностей: 0
   Всего связей: 0
   Всего фактов: 0



In [ ]:
# Health check всех сервисов
print("🏥 Проверка здоровья всех сервисов...\n")

services = [
    ("Working Memory", working_memory),
    ("Episodic Memory", episodic_memory),
    ("Semantic Memory", semantic_memory),
    ("Procedural Memory", procedural_memory),
    ("Facts Memory", facts_memory)
]

all_healthy = True
for name, service in services:
    health = await service.health_check()
    status = health.get('status', 'unknown')
    icon = "✅" if status == "healthy" else "❌"
    print(f"{icon} {name}: {status}")
    if status != "healthy":
        all_healthy = False
        if 'error' in health:
            print(f"   Ошибка: {health['error']}")

print("\n" + ("✅ Все сервисы работают корректно!" if all_healthy else "⚠️ Обнаружены проблемы в некоторых сервисах"))

🏥 Проверка здоровья всех сервисов...

✅ Working Memory: healthy
✅ Episodic Memory: healthy
✅ Semantic Memory: healthy
✅ Procedural Memory: healthy
✅ Facts Memory: healthy

✅ Все сервисы работают корректно!


## Заключение

Вы ознакомились со всеми возможностями Memory Agents:

✅ **Working Memory** - управление краткосрочным контекстом сессий  
✅ **Episodic Memory** - хранение и анализ эпизодов взаимодействия  
✅ **Semantic Memory** - долговременные знания с векторным поиском  
✅ **Procedural Memory** - процедуры и паттерны с отслеживанием успеха  
✅ **Facts Memory** - граф знаний с сущностями и связями  
✅ **Консолидация** - автоматическое извлечение знаний из эпизодов  
✅ **Интеграция** - совместная работа всех типов памяти  
✅ **Мониторинг** - детальные метрики и health checks  

### Следующие шаги:

1. Изучите документацию в `docs/` для более глубокого понимания
2. Посмотрите примеры интеграции в `examples/`
3. Адаптируйте систему под свои задачи
4. Экспериментируйте с различными конфигурациями памяти

### Полезные ссылки:

- [Архитектура](../docs/ARCHITECTURE.md)
- [API Reference](../docs/API_REFERENCE.md)
- [Примеры использования](../docs/EXAMPLES.md)
- [Развертывание](../docs/DEPLOYMENT.md)

In [ ]:
# Очистка: закрытие сессии
print("🧹 Завершение демонстрации...\n")

# Обновить контекст сессии перед закрытием
await working_memory.update_context(
    session_id=session_id,
    context_updates={
        "demo_completed": True,
        "completion_time": datetime.now().isoformat(),
        "all_features_demonstrated": True
    }
)

print("✅ Демонстрация успешно завершена!")
print(f"\n🎉 Спасибо за использование Memory Agents!")
print(f"⏰ Время завершения: {datetime.now()}")

Failed to update context for session session_demo_2025: Session session_demo_2025 not found


🧹 Завершение демонстрации...

✅ Демонстрация успешно завершена!

🎉 Спасибо за использование Memory Agents!
⏰ Время завершения: 2025-10-30 22:48:06.350964
